# 📊 Project 3: Derivable Judgement — A Statistical Decision-Making Model


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

df = pd.read_csv('data/health_records.csv')
df.head()

,record_id,age_group,age,weight,gender,region,smoking_status,exercise_frequency,bmi,blood_pressure,diabetes,hypertension,cholesterol_level,glucose_level,visit_date
0,R001,26-35,28,70,Male,North,Non-Smoker,Daily,22.5,118.0,False,False,180.0,90.0,2024-01-10
1,R002,46-60,52,90,Female,South,Smoker,Rarely,30.1,145.0,True,True,240.0,130.0,2024-01-11
2,R003,18-25,22,60,Female,East,Non-Smoker,Weekly,21.0,112.0,False,False,170.0,85.0,2024-01-12
3,R004,36-45,40,85,Male,West,Former Smoker,Weekly,27.3,135.0,True,False,210.0,115.0,2024-01-13
4,R005,60+,65,75,Female,North,Non-Smoker,Never,26.0,150.0,True,True,260.0,140.0,2024-01-14


In [3]:
# Dataset overview
df.dtypes

record_id                 str
age_group                 str
age                     int64
weight                  int64
gender                    str
region                    str
smoking_status            str
exercise_frequency        str
bmi                   float64
blood_pressure        float64
diabetes                 bool
hypertension             bool
cholesterol_level     float64
glucose_level         float64
visit_date                str
dtype: object

In [4]:
# Basic statistics
df.describe()

,age,weight,bmi,blood_pressure,cholesterol_level,glucose_level
count,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000
mean,42.920000,75.980000,25.982000,134.100000,217.10000,115.040000
std,17.189033,11.193092,3.657237,17.852285,41.79603,28.121428
min,19.000000,56.000000,19.800000,107.000000,161.00000,80.000000
25%,28.250000,68.000000,22.825000,118.000000,180.25000,90.250000
50%,41.500000,76.500000,26.650000,134.500000,210.00000,109.500000
75%,54.750000,85.000000,28.500000,150.000000,260.00000,141.500000
max,74.000000,95.000000,32.500000,164.000000,290.00000,168.000000


---
## 📘 Part A — Theoretical Foundation (Short Notes)


### 1. Inferential Statistics

Inferential statistics uses **sample data** to make predictions or draw conclusions about a **larger population**. Unlike descriptive statistics, it allows us to test hypotheses, estimate population parameters, and determine relationships between variables.

### 2. Hypothesis Testing

A method to decide whether sample evidence supports a claim.

- **H₀ (Null Hypothesis):** Assumes no effect / no difference.
- **H₁ (Alternative Hypothesis):** Claims a significant effect.
- **α (Significance Level):** Threshold for rejection (commonly 0.05).
- **Test Statistic:** Calculated value from data.
- **p-value:** Probability of observed result under H₀.

### 3. Confidence Interval & Critical Value

A **Confidence Interval (CI)** is a range likely to contain the true population parameter (e.g., 95% CI → 95% confident). **Critical Value** is the threshold from the distribution table (z or t) beyond which we reject H₀. Example: z-critical for 95% CI = **±1.96**.

### 4. p-value

The **p-value** is the probability of obtaining results as extreme as observed, assuming H₀ is true. - If **p ≤ α (0.05)** → Reject H₀ (statistically significant).
- If **p > α** → Fail to reject H₀.

### 5. Type I and Type II Errors

| Error | Type | Description |
|---|---|---|
| **Type I (α)** | False Positive | Rejecting H₀ when it is actually **true** |
| **Type II (β)** | False Negative | Failing to reject H₀ when H₁ is actually **true** |

> Trade-off: Reducing Type I error increases the risk of Type II error.

### 6. Brief Description of Statistical Tests

| Test | When to Use |
|---|---|
| **Z-test** | n > 30, population std known; tests means |
| **T-test** | n < 30 or unknown std; compares group means |
| **Chi-Square** | Tests association between categorical variables |
| **ANOVA** | Compares means across 3+ groups simultaneously |

### 7. Covariance

**Covariance** measures how two variables change together.

- **Positive** → both increase together.
- **Negative** → one increases as the other decreases.
- **Scale-dependent** → makes comparison across datasets difficult.

### 8. Correlation

**Correlation** is the standardized covariance, ranging from **−1 to +1**.

- r = +1 → Perfect positive relationship
- r = −1 → Perfect negative relationship
- r = 0 → No linear relationship

> Scale-independent and easier to interpret than covariance.

---
## 🔬 Part B — Data Analysis & Testing Tasks


### Task 1 — Formulate Hypotheses

**Hypothesis Set 1 – Smoking & Diabetes**
- **H₀:** Smoking has no effect on Diabetes prevalence.
- **H₁:** Smoking significantly affects Diabetes prevalence.

**Hypothesis Set 2 – Age Group & Glucose Level**
- **H₀:** Age group has no significant effect on Glucose/Hypertension rate.
- **H₁:** Older age groups have significantly higher Glucose/Hypertension rates.


### Task 2 — Confidence Intervals (Age, Weight, BMI)

In [5]:
def confidence_interval(data, confidence=0.95):
    n    = len(data)
    mean = np.mean(data)
    se   = stats.sem(data)
    h    = se * stats.t.ppf((1 + confidence) / 2, df=n - 1)
    return mean, mean - h, mean + h

results = {}
for col in ['age', 'weight', 'bmi']:
    mean, low, high = confidence_interval(df[col])
    results[col] = {'Mean': round(mean,2), 'CI Lower': round(low,2), 'CI Upper': round(high,2)}

ci_df = pd.DataFrame(results).T
print("95% Confidence Intervals:")
ci_df

95% Confidence Intervals:


,Mean,CI Lower,CI Upper
age,42.92,38.03,47.81
weight,75.98,72.80,79.16
bmi,25.98,24.94,27.02


### Task 3 & 4 — Critical Value, p-value & T-Test (Smoker vs Non-Smoker BMI)

In [6]:
smokers_bmi     = df[df['smoking_status'] == 'Smoker']['bmi']
non_smokers_bmi = df[df['smoking_status'] == 'Non-Smoker']['bmi']

n_s  = len(smokers_bmi)
n_ns = len(non_smokers_bmi)

t_stat, p_value    = stats.ttest_ind(smokers_bmi, non_smokers_bmi)
critical_value     = stats.t.ppf(0.975, df=n_s + n_ns - 2)

print(f"Smokers    : n={n_s},  Mean BMI = {smokers_bmi.mean():.2f}")
print(f"Non-Smokers: n={n_ns}, Mean BMI = {non_smokers_bmi.mean():.2f}")
print(f"\nT-Statistic    : {t_stat:.4f}")
print(f"Critical Value : ±{critical_value:.4f}")
print(f"P-Value        : {p_value:.4f}")
print(f"\n{'REJECT H0' if p_value < 0.05 else 'FAIL TO REJECT H0'} → "
      f"{'Significant difference in BMI between groups.' if p_value < 0.05 else 'No significant BMI difference.'}")

Smokers    : n=18,  Mean BMI = 28.72
Non-Smokers: n=24, Mean BMI = 23.37

T-Statistic    : 5.9956
Critical Value : ±2.0211
P-Value        : 0.0000

REJECT H0 → Significant difference in BMI between groups.


### Task 5 — Chi-Square Test (Smoking Status vs Diabetes)

In [7]:
contingency = pd.crosstab(df['smoking_status'], df['diabetes'])
print("Contingency Table:")
print(contingency)

chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
chi_critical = stats.chi2.ppf(0.95, df=dof)

print(f"\nChi-Square Statistic : {chi2:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"Critical Value       : {chi_critical:.4f}")
print(f"P-Value              : {p_chi:.4f}")
print(f"\n{'REJECT H0' if p_chi < 0.05 else 'FAIL TO REJECT H0'} → "
      f"{'Smoking is significantly associated with Diabetes.' if p_chi < 0.05 else 'No significant association found.'}")

Contingency Table:
diabetes        False  True 
smoking_status              
Former Smoker       2      6
Non-Smoker         20      4
Smoker              3     15

Chi-Square Statistic : 20.6667
Degrees of Freedom   : 2
Critical Value       : 5.9915
P-Value              : 0.0000

REJECT H0 → Smoking is significantly associated with Diabetes.


### Task 6 — One-Way ANOVA (Age Group vs Glucose Level)

In [8]:
groups = [grp['glucose_level'].values for _, grp in df.groupby('age_group')]

print("Group Means:")
for name, grp in df.groupby('age_group'):
    print(f"  {name:10s}: Mean Glucose = {grp['glucose_level'].mean():.2f}")

f_stat, p_anova = stats.f_oneway(*groups)
f_critical = stats.f.ppf(0.95, dfn=len(groups)-1, dfd=len(df)-len(groups))

print(f"\nF-Statistic  : {f_stat:.4f}")
print(f"F-Critical   : {f_critical:.4f}")
print(f"P-Value      : {p_anova:.4f}")
print(f"\n{'REJECT H0' if p_anova < 0.05 else 'FAIL TO REJECT H0'} → "
      f"{'Age groups significantly differ in Glucose levels.' if p_anova < 0.05 else 'No significant difference.'}")

Group Means:
  18-25     : Mean Glucose = 83.50
  26-35     : Mean Glucose = 91.70
  36-45     : Mean Glucose = 112.90
  46-60     : Mean Glucose = 131.80
  60+       : Mean Glucose = 155.30

F-Statistic  : 90.3385
F-Critical   : 2.5787
P-Value      : 0.0000

REJECT H0 → Age groups significantly differ in Glucose levels.


### Task 7 — Covariance and Correlation

In [9]:
pairs = [
    ('age',           'bmi',              'Age vs BMI'),
    ('bmi',           'blood_pressure',   'BMI vs Blood Pressure'),
    ('glucose_level', 'cholesterol_level','Glucose vs Cholesterol'),
]

rows = []
for x, y, label in pairs:
    cov  = np.cov(df[x], df[y])[0][1]
    corr, p = stats.pearsonr(df[x], df[y])
    rows.append({'Pair': label, 'Covariance': round(cov,4),
                 'Correlation (r)': round(corr,4), 'p-value': round(p,4),
                 'Interpretation': 'Strong' if abs(corr)>0.7 else 'Moderate' if abs(corr)>0.4 else 'Weak'})

pd.DataFrame(rows).set_index('Pair')

,Covariance,Correlation (r),p-value,Interpretation
Pair,,,,
Age vs BMI,42.8965,0.6824,0.0,Moderate
BMI vs Blood Pressure,50.0549,0.7667,0.0,Strong
Glucose vs Cholesterol,1167.7510,0.9935,0.0,Strong


### Task 8 — Results Summary (Accept / Reject H₀)

In [10]:
summary = pd.DataFrame([
    {'Test': 'T-Test',      'Variables': 'Smoker vs Non-Smoker BMI',    'α': 0.05, 'p-value': round(p_value,4),  'Decision': 'REJECT H0' if p_value < 0.05 else 'FAIL TO REJECT H0'},
    {'Test': 'Chi-Square',  'Variables': 'Smoking Status vs Diabetes',  'α': 0.05, 'p-value': round(p_chi,4),    'Decision': 'REJECT H0' if p_chi   < 0.05 else 'FAIL TO REJECT H0'},
    {'Test': 'ANOVA',       'Variables': 'Age Group vs Glucose Level',  'α': 0.05, 'p-value': round(p_anova,4),  'Decision': 'REJECT H0' if p_anova < 0.05 else 'FAIL TO REJECT H0'},
])
print("=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
summary

FINAL RESULTS SUMMARY


,Test,Variables,α,p-value,Decision
0,T-Test,Smoker vs Non-Smoker BMI,0.05,0.0,REJECT H0
1,Chi-Square,Smoking Status vs Diabetes,0.05,0.0,REJECT H0
2,ANOVA,Age Group vs Glucose Level,0.05,0.0,REJECT H0
